# 02 — Stage-1 特征分析报告解读

先决条件:已跑 `PYTHONPATH=src python scripts/run_analysis.py --product <PRODUCT>`,本 notebook 重新加载 `report/summary.csv` 做交互式解读(逐特征明细另见浏览器打开 `report/index.html`)。

xc 口径要点(configs/products/xc_*.yaml):
- `rank_weights` 抬高 lift/concentration、压低 missing_penalty —— 排序偏向「把正例排到前面」的特征,且不过度惩罚高缺失;
- `lift_keep_min: 1.2` 软门槛:IV 不达标但 top-K lift 强的特征仍保留;
- 特征是纯编号、无时间窗后缀 → `enable_window_family: false`,没有家族去重那一层;
- Stage-1 只做统计粗筛(`stage1_top_n: 350`),模型法精筛在 Stage-1.5(null importance,评审见 03)。

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, '../src')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Image, display
from wdm.config import load_config
from wdm.utils.paths import per_feature_dir, report_dir

PRODUCT = 'xc_resp_finish'

cfg = load_config(PRODUCT)
ana = cfg['analysis']
print('筛选口径: iv_min={0} | lift_keep_min={1} | psi_cutoff={2} | missing_rate_max={3} | stage1_top_n={4}'.format(
    ana.get('iv_min'), ana.get('lift_keep_min'), ana.get('psi_cutoff'),
    ana.get('missing_rate_max'), ana.get('stage1_top_n')))
print('rank_weights:', ana.get('rank_weights'))
summary = pd.read_csv(report_dir(cfg) / 'summary.csv')
kept = summary['auto_keep'] == True
print('summary:', summary.shape, '| auto_keep:', int(kept.sum()))
KEY = [c for c in ['feature', 'feature_cn', 'iv', 'lift_at_k', 'gini', 'concentration', 'psi',
                   'flag', 'missing_rate', 'corr_cluster', 'rank_score', 'auto_keep', 'drop_reason']
       if c in summary.columns]
summary.sort_values('rank_score', ascending=False)[KEY].head(15)

In [ ]:
# 保留/剔除概况:千维特征不逐行列出,看 drop_reason 分布 + 两组指标中位数对比
print('auto_keep:', summary['auto_keep'].value_counts().to_dict())
print()
print('drop_reason 分布:')
print(summary.loc[~kept, 'drop_reason'].value_counts())
metric_cols = [c for c in ['iv', 'lift_at_k', 'gini', 'psi', 'missing_rate'] if c in summary.columns]
print()
print('两组指标中位数:')
summary.groupby(kept)[metric_cols].median().round(4)

In [ ]:
# 正例导向检查:IV x lift@K 散点;右下 = IV 弱但 top-K lift 强,
# 是 lift_keep_min 软门槛专门救回的目标特征(xc 的筛选取向)
iv_min, lift_keep_min = ana.get('iv_min'), ana.get('lift_keep_min')
plt.figure(figsize=(7, 5))
plt.scatter(summary.loc[~kept, 'iv'], summary.loc[~kept, 'lift_at_k'], s=10, alpha=0.35, label='dropped')
plt.scatter(summary.loc[kept, 'iv'], summary.loc[kept, 'lift_at_k'], s=12, alpha=0.6, label='auto_keep')
if iv_min:
    plt.axvline(iv_min, color='gray', lw=0.8, ls='--')
if lift_keep_min:
    plt.axhline(lift_keep_min, color='red', lw=0.8, ls='--', label='lift_keep_min')
plt.xlabel('IV'); plt.ylabel('lift_at_k'); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()

if lift_keep_min and iv_min:
    rescued = summary[kept & (summary['iv'] < iv_min) & (summary['lift_at_k'] >= lift_keep_min)]
    print('lift 软门槛救回(iv < {0} 且 lift_at_k >= {1}): {2} 个'.format(
        iv_min, lift_keep_min, len(rescued)))
    display(rescued[KEY].sort_values('lift_at_k', ascending=False).head(15))

In [ ]:
# PSI 稳定性:flag 分布(stable < 0.10 <= shift < 0.25 <= broken)。
# 重点:auto_keep 且非 stable 的特征 → 人工确认漂移可接受,上线后重点监控(07)
print(summary['flag'].value_counts(dropna=False))
watch = summary[kept & (summary['flag'] != 'stable')]
print()
print('auto_keep 且 PSI 非 stable({0} 个):'.format(len(watch)))
watch[KEY].sort_values('psi', ascending=False).head(20)

In [ ]:
# 相关性去冗余:最大的几个簇 + 最大簇内成员(谁留谁走,看 rank_score)
if 'corr_cluster' in summary.columns and summary['corr_cluster'].notna().any():
    sizes = summary['corr_cluster'].value_counts()
    print('簇个数:', sizes.size)
    print('最大 10 个簇:')
    print(sizes.head(10))
    big = sizes.index[0]
    cols = [c for c in KEY if c != 'drop_reason']
    display(summary[summary['corr_cluster'] == big][cols]
            .sort_values('rank_score', ascending=False).head(15))
else:
    print('无相关性簇信息')

In [ ]:
# 头部特征的图(Stage-1 仅为 rank 前 per_feature_plot_top_n 个特征出图,默认 50)
TOP_N = 3
PLOTS = ['dist.png', 'woe.png', 'lift.png']
for feat in summary.loc[kept].sort_values('rank_score', ascending=False)['feature'].head(TOP_N):
    d = per_feature_dir(cfg, feat)
    if not d.is_dir():
        print(feat, ': 无 per-feature 图目录(超出 per_feature_plot_top_n?)')
        continue
    for name in PLOTS:
        p = d / name
        if p.is_file():
            print(feat, '-', name)
            display(Image(str(p)))